# Laptop Support RAG Assistant

This notebook builds and evaluates a Retrieval-Augmented Generation (RAG) pipeline for answering laptop support questions using three laptop manuals.

The pipeline includes document loading, text chunking, embeddings, vector storage, retrieval, prompt construction, local Ollama generation, evaluation, and vector store persistence.


In [2]:
!pip install -q pypdf sentence-transformers chromadb ollama

In [4]:
# Project Imports
import os
import json
import time
import shutil

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import chromadb
import ollama


c:\Users\Mohamed Ahmed\Desktop\Laptop_Support_RAG\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2.1 Load & Inspect

The dataset consists of three English laptop manuals in PDF format:

- Dell Latitude 5540 Owner's Manual
- Lenovo LOQ User Guide
- HP User Guide

The documents contain a total of 291 pages.

All three PDF files were successfully parsed using `pypdf`. No OCR was required because the documents contain extractable text.

In [7]:
from pypdf import PdfReader
import os

pdf_files = [
    "../data/pdfs/latitude-5540-owners-manual-en-us.pdf",
    "../data/pdfs/loq_x90_ug_en.pdf",
    "../data/pdfs/pdf_6883759_en-US-1_hp.pdf"
]


for file in pdf_files:
    print("=" * 70)
    print(f"File: {file}")

    reader = PdfReader(file)

    print(f"Number of pages: {len(reader.pages)}")

    # Try extracting text from the first page
    text = reader.pages[0].extract_text()

    if text and text.strip():
        print("Parsing status: SUCCESS")
        print("\nFirst 300 characters:")
        print(text[:300])
    else:
        print("Parsing status: FAILED / OCR may be needed")

File: ../data/pdfs/latitude-5540-owners-manual-en-us.pdf
Number of pages: 159
Parsing status: SUCCESS

First 300 characters:
Latitude 5540 
Owner's Manual
Regulatory Model: P127F
Regulatory Type: P127F001
September 2025
Rev. A05

File: ../data/pdfs/loq_x90_ug_en.pdf
Number of pages: 50
Parsing status: SUCCESS

First 300 characters:
User Guide
Lenovo LOQ 15AHP9, Lenovo LOQ 15ARP9, Lenovo LOQ 15IAX9, 
Lenovo LOQ 15IAX9I, and Lenovo LOQ 15IRX9
File: ../data/pdfs/pdf_6883759_en-US-1_hp.pdf
Number of pages: 82
Parsing status: SUCCESS

First 300 characters:
User Guide
SUMMARY
This guide provides information about components, network connection, power management, security, backing 
up, and more.


### Test: Verify Extracted Text

Before applying the chunking strategy, we test the extracted text from the Dell manual.

This test prints the first 1000 characters to verify that the PDF text was extracted correctly and is ready for the chunking step.

In [8]:
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text = text + page_text

    return text

In [10]:
dell_text = extract_text_from_pdf(
    "../data/pdfs/latitude-5540-owners-manual-en-us.pdf"
)

print(dell_text[:1000]) #1000 Characters

Latitude 5540 
Owner's Manual
Regulatory Model: P127F
Regulatory Type: P127F001
September 2025
Rev. A05
Notes, cautions, and warnings
NOTE:  A NOTE indicates important information that helps you make better use of your product.
CAUTION:  A CAUTION indicates either potential damage to hardware or loss of data and tells you how to avoid 
the problem.
© 2023-2024 Dell Inc. or its subsidiaries. All rights reserved. Dell Technologies, Dell, and other trademarks are trademarks of Dell Inc. or its 
subsidiaries. Other trademarks may be trademarks of their respective owners. Chapter 1: Views of Latitude 5540 .................................................................................................8
Right ....................................................................................................................................................................................... 8
Left.......


## 2.2 Chunking Strategy

The extracted text is divided into smaller chunks before creating embeddings and storing the data in the vector store.

### Chunking Method

We use fixed-size character-based chunking with:

- Chunk size: 1000 characters
- Chunk overlap: 200 characters

The chunk size determines the approximate amount of text included in each chunk.

The overlap keeps part of the previous chunk in the next chunk. This helps preserve context when an important piece of information is located near the boundary between two chunks.


A chunk size of 1000 characters provides enough context for typical laptop support instructions while keeping the retrieved context manageable.

An overlap of 200 characters helps preserve information that may span across chunk boundaries and reduces the chance of losing important context between consecutive chunks.

The resulting dataset contains 582 chunks.

In [11]:
def create_chunks(text, chunk_size, chunk_overlap):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start = end - chunk_overlap

    return chunks

In [12]:
# Extract the text from the three PDF files

dell_text = extract_text_from_pdf(
    "latitude-5540-owners-manual-en-us.pdf"
)

lenovo_text = extract_text_from_pdf(
    "loq_x90_ug_en.pdf"
)

hp_text = extract_text_from_pdf(
    "pdf_6883759_en-US-1_hp.pdf"
)

In [ ]:
# Create chunks for each document

dell_chunks = create_chunks(
    dell_text,
    1000,
    200
)

lenovo_chunks = create_chunks(
    lenovo_text,
    1000,
    200
)

hp_chunks = create_chunks(
    hp_text,
    1000,
    200
)

In [15]:
# Test for chunks
print("Dell chunks:", len(dell_chunks))
print("Lenovo chunks:", len(lenovo_chunks))
print("HP chunks:", len(hp_chunks))

Dell chunks: 276
Lenovo chunks: 96
HP chunks: 210


In [16]:
# Test the first chunk

print(dell_chunks[0])

Latitude 5540 
Owner's Manual
Regulatory Model: P127F
Regulatory Type: P127F001
September 2025
Rev. A05
Notes, cautions, and warnings
NOTE:  A NOTE indicates important information that helps you make better use of your product.
CAUTION:  A CAUTION indicates either potential damage to hardware or loss of data and tells you how to avoid 
the problem.
© 2023-2024 Dell Inc. or its subsidiaries. All rights reserved. Dell Technologies, Dell, and other trademarks are trademarks of Dell Inc. or its 
subsidiaries. Other trademarks may be trademarks of their respective owners. Chapter 1: Views of Latitude 5540 .................................................................................................8
Right ....................................................................................................................................................................................... 8
Left.......


In [17]:
# Check the size of the first chunk

print("First chunk length:", len(dell_chunks[0]))

First chunk length: 1000


In [18]:
# Test the overlap between the first two chunks

first_chunk_end = dell_chunks[0][-200:]
second_chunk_start = dell_chunks[1][:200]

print("Last 200 characters of Chunk 1:")
print(first_chunk_end)

print("\nFirst 200 characters of Chunk 2:")
print(second_chunk_start)

print("\nAre they the same?")
print(first_chunk_end == second_chunk_start)

Last 200 characters of Chunk 1:
ht ....................................................................................................................................................................................... 8
Left.......

First 200 characters of Chunk 2:
ht ....................................................................................................................................................................................... 8
Left.......

Are they the same?
True


## 2.3 Embeddings & Vector Store

After splitting the documents into chunks, the next step is to convert the text chunks into numerical vector representations called embeddings.

Embeddings represent the semantic meaning of the text. Chunks with similar meanings should have similar vector representations.

For this project, we use an embedding model to convert the 582 text chunks into vectors.

These vectors are stored in a ChromaDB vector store.

### Pipeline

The process is:

1. Extracted text
2. Text chunks
3. Embeddings
4. ChromaDB vector store

The vector store will later be used during retrieval to find the most relevant chunks for a user's question.

The ChromaDB collection is persisted so that it can be reused by the backend without rebuilding it every time.

In [5]:
# Load the embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# Test the embedding model
test_text = "How do I replace the laptop battery?"

test_embedding = embedding_model.encode(
    test_text
)

print("Test embedding:")
print(test_embedding)


chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)



collection = chroma_client.get_or_create_collection(
    name="laptop_manuals"
)

print("ChromaDB collection is ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16027.21it/s]


Test embedding:
[-5.35962312e-03  1.47060499e-01 -5.59836403e-02 -2.65599489e-02
  5.98186180e-02  2.72413921e-02 -2.29204502e-02  3.76523240e-03
  6.81180656e-02  4.94025871e-02  1.15956189e-02 -5.91300875e-02
 -5.10264114e-02  5.83465174e-02  4.17543165e-02 -1.89908911e-02
 -1.67404898e-02 -6.93770358e-03  2.54007746e-02 -4.78771478e-02
  1.46792792e-02 -5.83965285e-03  2.45427135e-02 -1.98831111e-02
  9.65391695e-02 -3.37902196e-02  1.96462590e-02 -1.69306099e-02
 -8.95410106e-02 -2.12779688e-03  1.28873419e-02  6.30773157e-02
 -8.18673987e-03  5.85193485e-02 -4.66576125e-03  3.88967693e-02
  2.43075378e-02  1.60688888e-02  8.89183208e-02 -5.42115867e-02
 -1.27434116e-02  1.42841758e-02  3.42029668e-02 -5.12089441e-03
  6.13189898e-02  3.67119424e-02  2.39986684e-02  3.97328548e-02
  1.34614766e-01  4.64352593e-02  5.10719791e-02 -4.96004336e-02
 -4.40536365e-02 -2.54789571e-04 -1.61019620e-02 -1.89688019e-02
  7.44722262e-02  4.04996723e-02  5.75707927e-02 -1.29964575e-02
  3.74647

In [6]:
# Check the number of stored chunks

print("Number of chunks:", collection.count())

Number of chunks: 582


### Creating Embeddings and Storing Them in ChromaDB

The 582 text chunks are converted into vector embeddings using the selected embedding model.

Each chunk is stored together with its embedding and metadata identifying its source document.

The embeddings and documents are added to the ChromaDB collection so that they can later be searched using semantic similarity.

Each stored item contains:

- A unique ID
- The original text chunk
- Its embedding vector
- Metadata identifying the source document

In [ ]:
# Combine all chunks into one list

all_chunks = []

all_chunks.extend(dell_chunks)
all_chunks.extend(lenovo_chunks)
all_chunks.extend(hp_chunks)


# Create a list of source names

sources = []

sources.extend(["Dell"] * len(dell_chunks))
sources.extend(["Lenovo"] * len(lenovo_chunks))
sources.extend(["HP"] * len(hp_chunks))


# Create unique IDs for all chunks

chunk_ids = []

for i in range(len(all_chunks)):
    chunk_ids.append(f"chunk_{i}")


# Create embeddings for all chunks

embeddings = embedding_model.encode(
    all_chunks,
    show_progress_bar=True
)


# Add chunks, embeddings, IDs, and metadata to ChromaDB

collection.upsert(
    ids=chunk_ids,
    documents=all_chunks,
    embeddings=embeddings.tolist(),
    metadatas=[
        {"source": source}
        for source in sources
    ]
)


print("Total chunks stored:", len(all_chunks))
print("Embeddings stored successfully.")

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Total chunks stored: 582
Embeddings stored successfully.


In [24]:
# Check the number of stored documents

stored_data = collection.get()

print("Documents stored in ChromaDB:", len(stored_data["ids"]))

Documents stored in ChromaDB: 582


## 2.4 Retrieval & Prompting

The retrieval component is responsible for finding the most relevant document chunks for a user's question.

The question is converted into an embedding using the same embedding model used for the document chunks.

The question embedding is then compared with the embeddings stored in ChromaDB.

The most relevant chunks are retrieved and used as context for the language model.

### Retrieval Pipeline

1. User asks a question.
2. The question is converted into an embedding.
3. ChromaDB searches for the most similar chunks.
4. The top relevant chunks are returned.
5. The retrieved chunks are used as context for the final answer.

This approach helps the language model generate answers based on the information contained in the laptop manuals.

In [7]:
def retrieve_chunks(question, number_of_results=5):

    # Convert the user's question into an embedding
    question_embedding = embedding_model.encode(
        question
    )

    # Search ChromaDB for the most relevant chunks
    results = collection.query(
        query_embeddings=[
            question_embedding.tolist()
        ],
        n_results=number_of_results
    )

    return results

In [8]:
results = retrieve_chunks(
    "How do I troubleshoot power problems?"
)
print(results)

{'ids': [['chunk_485', 'chunk_270', 'chunk_486', 'chunk_492', 'chunk_322']], 'embeddings': None, 'documents': [['n.\n1. Save your work and close all open programs.\n32 Chapter 6  Managing power2. Select the Start button, select the Power icon, and then select Shut down.\nIf the computer is unresponsive and you are unable to use the preceding shutdown procedures, try the \nfollowing emergency procedures in the sequence provided:\n● Press ctrl+alt+delete, select the Power icon, and then select Shut down.\n● Press and hold the power button for at least 10 seconds.\n● If your computer has a user-replaceable battery (select products only), disconnect the computer \nfrom external power, and then remove the battery.\nUsing the Power icon\nDifferent Power icons indicate whether the computer is running on battery or external power. Placing the \nmouse pointer over the icon reveals a message if the battery has reached a low or critical battery level.\nThe Power icon \n  is located on the Windows

In [9]:
def display_retrieved_chunks(results):

    # Get the retrieved documents
    documents = results["documents"][0]

    # Get the source of each document
    metadatas = results["metadatas"][0]

    # Get the distance of each document
    distances = results["distances"][0]

    # Display each retrieved chunk
    for i in range(len(documents)):

        print("=" * 70)
        print(f"Chunk {i + 1}")

        print("Source:", metadatas[i]["source"])

        print("Distance:", distances[i])

        print("\nText:")
        print(documents[i])

display_retrieved_chunks(results)

Chunk 1
Source: HP
Distance: 0.950758159160614

Text:
n.
1. Save your work and close all open programs.
32 Chapter 6  Managing power2. Select the Start button, select the Power icon, and then select Shut down.
If the computer is unresponsive and you are unable to use the preceding shutdown procedures, try the 
following emergency procedures in the sequence provided:
● Press ctrl+alt+delete, select the Power icon, and then select Shut down.
● Press and hold the power button for at least 10 seconds.
● If your computer has a user-replaceable battery (select products only), disconnect the computer 
from external power, and then remove the battery.
Using the Power icon
Different Power icons indicate whether the computer is running on battery or external power. Placing the 
mouse pointer over the icon reveals a message if the battery has reached a low or critical battery level.
The Power icon 
  is located on the Windows taskbar. The Power icon allows you to quickly access 
power settings an

### Retrieval Testing

To evaluate the retrieval component, we test it using 10 different questions related to common laptop support tasks.

The questions cover different topics such as battery, connectivity, charging, display, BIOS, maintenance, recovery, and hardware components.

For each question, the system retrieves the most relevant document chunks from ChromaDB.

The retrieved chunks are inspected to verify that they are related to the user's question.

In [10]:
# Create 10 sample questions for testing the retrieval system

test_questions = [
    "How do I replace the battery?",
    "How do I connect to a Wi-Fi network?",
    "How do I turn on Bluetooth?",
    "How do I charge the laptop?",
    "How do I connect an external monitor?",
    "How do I troubleshoot power problems?",
    "How do I update the BIOS?",
    "How do I clean the laptop?",
    "How do I recover or reset the laptop?",
    "How do I install or remove a component?"
]


# Test the retrieval system with each question

for question in test_questions:

    print("=" * 80)
    print("Question:", question)

    # Retrieve the 3 most relevant chunks
    results = retrieve_chunks(
        question,
        3
    )

    # Get the retrieved documents
    documents = results["documents"][0]

    # Get the source of each document
    metadatas = results["metadatas"][0]

    # Display the retrieved chunks
    for i in range(len(documents)):

        print(f"\nChunk {i + 1}")
        print("Source:", metadatas[i]["source"])
        print("Text:", documents[i][:500])

Question: How do I replace the battery?

Chunk 1
Source: Dell
Text: tery cable to the battery.
5. Remove the battery cable from the routing guides on the battery.
6. Disconnect the battery cable from the connector on the battery.
7. Remove the battery cable away from the battery.
Installing the battery
CAUTION:  The information in this section is intended for authorized service technicians only.
Prerequisites
If you are replacing a component, remove the existing component before performing the installation process.
Removing and installing Field Replaceable Units

Chunk 2
Source: Dell
Text:  Before working inside your computer .
2. Remove the SIM card .
3. Remove the base cover .
About this task
CAUTION:  Removing the battery resets the BIOS setup program’s settings to default. It is recommended that 
you note the BIOS setup program’s settings before removing the battery.
The following image(s) indicate the location of the battery and provide a visual representation of the removal proce

### Prompt Construction

The retrieved document chunks are combined into a context that is provided to the language model together with the user's question.

The prompt instructs the language model to answer using only the provided context.

This helps reduce unsupported answers and keeps the response grounded in the laptop manuals.

The prompt also asks the model to mention the source of the retrieved information when providing an answer.

In [11]:
def create_prompt(question, results):

    # Get the retrieved document chunks
    documents = results["documents"][0]

    # Get the source of each retrieved chunk
    metadatas = results["metadatas"][0]

    # Start building the context
    context = ""

    # Add every retrieved chunk to the context
    for i in range(len(documents)):

        context = context + f"""
Source: {metadatas[i]["source"]}

Document Chunk:
{documents[i]}

"""


    # Create the final prompt
    prompt = f"""
You are a laptop support assistant.

Your job is to answer the user's question using ONLY the information
provided in the document context below.

Rules:

1. Do not use outside knowledge.
2. Do not invent or assume information.
3. If the answer is not available in the provided context,
   say: "The information is not available in the provided laptop manuals."
4. Give a clear and concise answer.
5. When the context provides step-by-step instructions,
   present them as numbered steps.
6. Mention the source document used for the answer.
7. If multiple sources are relevant, mention all relevant sources.

Document Context:
{context}

User Question:
{question}

Answer:
"""


    # Return the completed prompt
    return prompt

In [12]:
prompt = create_prompt(
    "How do I troubleshoot power problems?",
    results
)

print(prompt[:3000])


You are a laptop support assistant.

Your job is to answer the user's question using ONLY the information
provided in the document context below.

Rules:

1. Do not use outside knowledge.
2. Do not invent or assume information.
3. If the answer is not available in the provided context,
   say: "The information is not available in the provided laptop manuals."
4. Give a clear and concise answer.
5. When the context provides step-by-step instructions,
   present them as numbered steps.
6. Mention the source document used for the answer.
7. If multiple sources are relevant, mention all relevant sources.

Document Context:

Source: Dell

Document Chunk:
k and provide a visual representation of the removal procedure.
Figure 41. Image: Heatsink
Steps
1. Loosen the seven captive screws that secure the heat sink to the system board.
NOTE:  Loosen the captive screws in the reverse sequential order mentioned on the heat sink [7 > 6 > 5 > 4 > 3 > 2 > 1].
NOTE:  The number of screws varies depend

In [13]:
# Generate an answer using Ollama

def generate_answer(question):

    # Retrieve the top 5 relevant chunks
    results = retrieve_chunks(question, 5)

    # Create the prompt using the retrieved context
    prompt = create_prompt(question, results)

    # Generate the answer using the local Ollama model
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [14]:
answer = generate_answer(
    "How do I troubleshoot power problems?"
)

print(answer)

To troubleshoot power problems, follow these steps:

1. Save your work and close all open programs (Source: HP).
2. Select the Start button, select the Power icon, and then select Shut down (Source: HP).
3. If the computer is unresponsive, try pressing ctrl+alt+delete, select the Power icon, and then select Shut down (Source: HP).
4. If the computer is unresponsive, press and hold the power button for at least 10 seconds (Source: HP).
5. If the computer has a user-replaceable battery, disconnect the computer from external power and remove the battery (Source: HP).
6. Turn off the computer and modem (Source: Dell).
7. Wait for 30 seconds.
8. Turn on the modem, wireless router, and then the computer (Source: Dell).
9. Alternatively, try "Drain residual flea power" by performing a hard reset (Source: HP).
10. Check the battery percentage by placing the mouse pointer over the Power icon (Source: HP).
11. If necessary, view power and battery settings by right-clicking the Power icon and sel

In [15]:
test_question = "How do I troubleshoot power problems?"

test_answer = generate_answer(test_question)

print("Question:")
print(test_question)

print("\nAnswer:")
print(test_answer)

Question:
How do I troubleshoot power problems?

Answer:
To troubleshoot power problems, follow these steps:

1. Save your work and close all open programs. (Source: HP, Chapter 6 Managing power)
2. Select the Start button, select the Power icon, and then select Shut down.
   If the computer is unresponsive, try pressing ctrl+alt+delete, select the Power icon, and then select Shut down. (Source: HP, Chapter 6 Managing power)
   If the computer is unresponsive, press and hold the power button for at least 10 seconds. (Source: HP, Chapter 6 Managing power)
   If the computer has a user-replaceable battery, disconnect the computer from external power and then remove the battery. (Source: HP, Chapter 6 Managing power)
3. If you are unable to use the above shutdown procedures, try resetting the Wi-Fi device by following the steps below:
   1. Turn off the computer. (Source: Dell, Wi-Fi power cycle)
   2. Turn off the modem.
   3. Turn off the wireless router.
   4. Wait for 30 seconds.
   5

## 2.6 Evaluation

The RAG system is evaluated using 10 test questions covering different laptop support topics.

For each question, we check:

- Whether the retrieved context is relevant to the question.
- Whether the generated answer is grounded in the retrieved context.
- Whether the answer contains unsupported or hallucinated information.
- Any failure cases observed during testing.

The evaluation results are summarized in a table.

In [16]:
# Evaluation questions

evaluation_questions = [
    "How do I replace the battery?",
    "How do I connect to a Wi-Fi network?",
    "How do I turn on Bluetooth?",
    "How do I charge the laptop?",
    "How do I connect an external monitor?",
    "How do I troubleshoot power problems?",
    "How do I update the BIOS?",
    "How do I clean the laptop?",
    "How do I recover or reset the laptop?",
    "How do I install or remove a component?"
]

print("Number of evaluation questions:", len(evaluation_questions))

Number of evaluation questions: 10


In [17]:
# Generate answers for all evaluation questions

evaluation_answers = []

for question in evaluation_questions:

    print("=" * 80)
    print("Question:", question)


    answer = generate_answer(
        question
    )

    evaluation_answers.append(answer)

    print("\nAnswer:")
    print(answer)

Question: How do I replace the battery?

Answer:
To replace the battery, follow these steps:

1. Disconnect the battery cable from the system board (if not disconnected earlier). (Figure 28)
2. Loosen the five captive screws that secure the battery to the palm-rest assembly. (Figure 30)
3. Lift the battery off the palm-rest assembly.
4. Flip the battery and peel the tape that adheres the battery cable to the battery. (Figure 30)
5. Remove the battery cable from the routing guides on the battery.
6. Disconnect the battery cable from the connector on the battery.
7. Remove the battery cable away from the battery.

Source: Dell, Document Chunk: Removing and installing Field Replaceable Units (FRUs)
Question: How do I connect to a Wi-Fi network?

Answer:
To connect to a Wi-Fi network, follow these steps:

1. Select the network icon on the bottom right of your display. (Source: Lenovo User Guide)
2. Select an available network, and then select Connect. If you want to be automatically connec

## 2.6 Evaluation results
The RAG system was evaluated using 10 test questions covering common laptop support tasks.

For each question, we evaluated:

- Retrieval relevance: whether the retrieved document context was relevant to the question.
- Answer groundedness: whether the generated answer was supported by the retrieved context.
- Failure cases: situations where retrieval or answer generation was incomplete or less specific.

The evaluation showed that the system generally retrieves relevant information and produces grounded answers, while a small number of cases require further improvement.

In [18]:
# Evaluation results based on manual inspection of the 10 test questions

evaluation_results = [
    {
        "question": "How do I replace the battery?",
        "retrieval_relevant": "Yes",
        "answer_grounded": "Yes",
        "notes": "Clear Dell battery removal and installation steps."
    },

    {
        "question": "How do I connect to a Wi-Fi network?",
        "retrieval_relevant": "Yes",
        "answer_grounded": "Yes",
        "notes": "Relevant information retrieved from Lenovo and HP."
    },

    {
        "question": "How do I turn on Bluetooth?",
        "retrieval_relevant": "Partial",
        "answer_grounded": "Yes",
        "notes": "The answer was grounded, but some retrieved context was less relevant."
    },

    {
        "question": "How do I charge the laptop?",
        "retrieval_relevant": "Yes",
        "answer_grounded": "Yes",
        "notes": "Relevant HP charging information was retrieved."
    },

    {
        "question": "How do I connect an external monitor?",
        "retrieval_relevant": "Yes",
        "answer_grounded": "Yes",
        "notes": "Relevant HP HDMI and display information was retrieved."
    },

    {
        "question": "How do I troubleshoot power problems?",
        "retrieval_relevant": "Partial",
        "answer_grounded": "Partial",
        "notes": "The retrieved context mixed general power troubleshooting with a Dell Wi-Fi power-cycle procedure."
    },

    {
        "question": "How do I update the BIOS?",
        "retrieval_relevant": "Partial",
        "answer_grounded": "Partial",
        "notes": "The answer used information from the retrieved context, but the context contained different BIOS-related procedures."
    },

    {
        "question": "How do I clean the laptop?",
        "retrieval_relevant": "Yes",
        "answer_grounded": "Yes",
        "notes": "Relevant HP cleaning instructions were retrieved."
    },

    {
        "question": "How do I recover or reset the laptop?",
        "retrieval_relevant": "Yes",
        "answer_grounded": "Yes",
        "notes": "Relevant HP recovery and reset information was retrieved."
    },

    {
        "question": "How do I install or remove a component?",
        "retrieval_relevant": "Yes",
        "answer_grounded": "Yes",
        "notes": "Relevant Dell component removal and installation information was retrieved."
    }
]


# Display the evaluation results

for result in evaluation_results:

    print("=" * 80)
    print("Question:", result["question"])
    print("Retrieval relevant:", result["retrieval_relevant"])
    print("Answer grounded:", result["answer_grounded"])
    print("Notes:", result["notes"])

Question: How do I replace the battery?
Retrieval relevant: Yes
Answer grounded: Yes
Notes: Clear Dell battery removal and installation steps.
Question: How do I connect to a Wi-Fi network?
Retrieval relevant: Yes
Answer grounded: Yes
Notes: Relevant information retrieved from Lenovo and HP.
Question: How do I turn on Bluetooth?
Retrieval relevant: Partial
Answer grounded: Yes
Notes: The answer was grounded, but some retrieved context was less relevant.
Question: How do I charge the laptop?
Retrieval relevant: Yes
Answer grounded: Yes
Notes: Relevant HP charging information was retrieved.
Question: How do I connect an external monitor?
Retrieval relevant: Yes
Answer grounded: Yes
Notes: Relevant HP HDMI and display information was retrieved.
Question: How do I troubleshoot power problems?
Retrieval relevant: Partial
Answer grounded: Partial
Notes: The retrieved context mixed general power troubleshooting with a Dell Wi-Fi power-cycle procedure.
Question: How do I update the BIOS?
Retri

In [20]:
# Create an evaluation table

import pandas as pd

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,retrieval_relevant,answer_grounded,notes
0,How do I replace the battery?,Yes,Yes,Clear Dell battery removal and installation st...
1,How do I connect to a Wi-Fi network?,Yes,Yes,Relevant information retrieved from Lenovo and...
2,How do I turn on Bluetooth?,Partial,Yes,"The answer was grounded, but some retrieved co..."
3,How do I charge the laptop?,Yes,Yes,Relevant HP charging information was retrieved.
4,How do I connect an external monitor?,Yes,Yes,Relevant HP HDMI and display information was r...
5,How do I troubleshoot power problems?,Partial,Partial,The retrieved context mixed general power trou...
6,How do I update the BIOS?,Partial,Partial,The answer used information from the retrieved...
7,How do I clean the laptop?,Yes,Yes,Relevant HP cleaning instructions were retrieved.
8,How do I recover or reset the laptop?,Yes,Yes,Relevant HP recovery and reset information was...
9,How do I install or remove a component?,Yes,Yes,Relevant Dell component removal and installati...


In [21]:
# Evaluation summary

print("Total questions:", len(evaluation_df))
print("Retrieval relevant:", 
      (evaluation_df["retrieval_relevant"] == "Yes").sum())

print("Retrieval partial:",
      (evaluation_df["retrieval_relevant"] == "Partial").sum())

print("Answers grounded:",
      (evaluation_df["answer_grounded"] == "Yes").sum())

print("Answers partial:",
      (evaluation_df["answer_grounded"] == "Partial").sum())

Total questions: 10
Retrieval relevant: 7
Retrieval partial: 3
Answers grounded: 8
Answers partial: 2


### Evaluation Summary

The evaluation was conducted using 10 laptop support questions covering common tasks such as battery replacement, Wi-Fi, Bluetooth, charging, external monitors, power troubleshooting, BIOS updates, cleaning, recovery, and component installation.

The evaluation results showed that:

- 7 out of 10 questions retrieved relevant context.
- 3 out of 10 questions had partially relevant retrieval results.
- 8 out of 10 answers were considered grounded in the retrieved documentation.
- 2 out of 10 answers were partially grounded.

The main observed issues were:

1. Some retrieved results contained less relevant chunks.
2. Broad questions sometimes retrieved information related to a specific component or procedure.
3. Some questions, such as power troubleshooting and BIOS updates, retrieved multiple related procedures, which affected the completeness and focus of the final answer.

These cases were used as failure cases for identifying possible RAG improvements.

### Retrieval Improvement

During the initial evaluation, the question:

**"How do I troubleshoot power problems?"**

did not produce a sufficiently detailed answer when only the top 3 retrieved chunks were used.

To improve retrieval coverage, the number of retrieved chunks was increased from **3 to 5**.

After increasing `top_k` to 5, additional context from the laptop manuals was retrieved. The generated answer provided useful troubleshooting procedures, although some retrieved information was less relevant to the general power troubleshooting question.

This shows that retrieving additional context can improve retrieval coverage when relevant information is not included in the initial top results. However, retrieving more chunks can also introduce less relevant information into the prompt.


### Updated Configuration

The retrieval configuration was updated during evaluation:

- Initial `top_k`: 3
- Improved `top_k`: 5
- Number of evaluation questions: 10
- Embedding model: `all-MiniLM-L6-v2`
- Vector store: ChromaDB

The change from `top_k = 3` to `top_k = 5` was tested using the power troubleshooting question.

The larger retrieval context provided additional information from the manuals and improved retrieval coverage compared with the initial configuration.

### Failure Cases and Limitations

The evaluation identified several retrieval limitations.

For the power troubleshooting question, increasing `top_k` from 3 to 5 provided additional context and improved retrieval coverage. However, some of the retrieved information was related to a Dell Wi-Fi power-cycle procedure rather than general power troubleshooting. Therefore, both retrieval relevance and answer grounding were considered partial for this question.

For the Bluetooth question, the final answer was grounded in the provided documentation, but some of the retrieved context was less relevant.

The BIOS update question also retrieved multiple BIOS-related procedures from the manuals. Although the answer used information from the retrieved documentation, the context contained different procedures, resulting in a partially grounded evaluation.

These cases show that retrieval quality directly affects the quality and focus of the generated answer. Increasing `top_k` can improve coverage, but retrieving too many chunks may introduce unnecessary or less relevant information into the prompt.

Possible future improvements include better chunking strategies, metadata filtering by laptop manufacturer or model, query reformulation, and more advanced retrieval or reranking methods.

Overall, the evaluation demonstrates that the system can answer a range of laptop support questions using information grounded in the provided manuals, while also highlighting areas for further retrieval improvement.


## 2.7 Export

The vector store and its configuration are prepared for use by the backend.

The persisted ChromaDB vector store contains the document chunks, embeddings, and source metadata generated during the RAG pipeline.

The backend can load this persisted vector store and perform retrieval without rebuilding the embeddings from the original PDF documents.

### Exported Components

* Persisted ChromaDB vector store
* Embedding model configuration
* Collection name
* Retrieval configuration
* Source document metadata

The exported components will be used by the FastAPI backend in the next stage of the project.


In [65]:
import os

# Check that the ChromaDB directory exists
if os.path.exists("chroma_db"):
    print("ChromaDB directory found.")
else:
    print("ChromaDB directory was not found.")

print("Documents stored in ChromaDB:", collection.count())

# Display the collection name
print("Collection name:", collection.name)

ChromaDB directory found.
Documents stored in ChromaDB: 582
Collection name: laptop_manuals


In [64]:
# Configuration used by the RAG pipeline

rag_config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_store": "ChromaDB",
    "collection_name": "laptop_manuals",
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "top_k": 5
}

print("RAG configuration:")
for key, value in rag_config.items():
    print(f"{key}: {value}")

RAG configuration:
embedding_model: all-MiniLM-L6-v2
vector_store: ChromaDB
collection_name: laptop_manuals
chunk_size: 1000
chunk_overlap: 200
top_k: 5


In [66]:
import json

# Save the RAG configuration
with open("rag_config.json", "w") as file:
    json.dump(rag_config, file, indent=4)

print("rag_config.json saved successfully.")

rag_config.json saved successfully.


In [67]:
# Verify that the configuration file exists
if os.path.exists("rag_config.json"):
    print("Configuration file exported successfully.")
else:
    print("Configuration file was not created.")

Configuration file exported successfully.


In [68]:
import chromadb

# Open the persisted ChromaDB from the saved directory
test_client = chromadb.PersistentClient(
    path="chroma_db"
)

# Load the existing collection
test_collection = test_client.get_collection(
    name="laptop_manuals"
)

# Check the stored documents
print("Collection loaded successfully.")
print("Collection name:", test_collection.name)
print("Number of stored documents:", test_collection.count())

Collection loaded successfully.
Collection name: laptop_manuals
Number of stored documents: 582


In [69]:
import shutil
import os

# Create a ZIP file containing the persisted ChromaDB
shutil.make_archive(
    "chroma_db_export",
    "zip",
    "chroma_db"
)

# Check that the ZIP file was created
if os.path.exists("chroma_db_export.zip"):
    print("ChromaDB export created successfully.")
    print("File: chroma_db_export.zip")
else:
    print("Export failed.")

ChromaDB export created successfully.
File: chroma_db_export.zip


In [70]:
from google.colab import files

# Download the persisted ChromaDB as a ZIP file
files.download("chroma_db_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [71]:
files.download("rag_config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>